In [ ]:
#import image test
import sys
from pathlib import Path 
from PIL import Image

project_root = Path(r"D:\university\Resesarch")
if str(project_root) not in sys.path:
    sys.path.insert(0,str(project_root / "test"))

print(f"Working directory: {Path.cwd()}")

image_path = Path(r"D:\university\Resesarch\J26-SE-334\Component3(local vision oracle)\notebooks\opening_screen.png")
print(f"Loadin image from: {image_path}")

img = Image.open(image_path).convert("RGB")
display(img.resize((300, int(300 * img.height / img.width))))

In [7]:
#test run raw input 1
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch

model_path = r"D:\university\Resesarch\RD\models\Qwen2-VL-2B-Instruct"

print("Loading model on GPU...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,   # ← faster on GPU
    device_map="cuda"              # ← use your RTX 3050
)
processor = AutoProcessor.from_pretrained(model_path)

img = Image.open(r"D:\university\Resesarch\J26-SE-334\Component3(local vision oracle)\notebooks\opening_screen.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "What UI defect or bug is on this screen?"}
        ]
    }
]

print("Processing...")
text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[img], return_tensors="pt").to("cuda")

print("Generating...")
output_ids = model.generate(**inputs, max_new_tokens=100)
result = processor.batch_decode(output_ids, skip_special_tokens=True)
print(result[0])

Loading model on GPU...


Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.48s/it]


Processing...
Generating...
system
You are a helpful assistant.
user
What UI defect or bug is on this screen?
assistant
The UI defect or bug on this screen is that the "Book Energy Slot" button is not clearly labeled or highlighted. The text "Book Energy Slot" is not bolded or in a larger font size, making it difficult to distinguish from other text on the screen. Additionally, the button is not clearly separated from the other options, which could lead to confusion when selecting this feature.


In [5]:
#test run raw input #2
img = Image.open(r"D:\university\Resesarch\J26-SE-334\Component3(local vision oracle)\notebooks\opening_screen.png")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": img},
            {"type": "text", "text": "Describe everything you see on this screen in detail."}
        ]
    }
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[img], return_tensors="pt").to("cuda")

output_ids = model.generate(**inputs, max_new_tokens=200)
result = processor.batch_decode(output_ids, skip_special_tokens=True)
print(result[0])

system
You are a helpful assistant.
user
Describe everything you see on this screen in detail.
assistant
The image is a screenshot of a mobile application interface, specifically for a microgrid management system. Here is a detailed description of the various components and functionalities visible in the screenshot:

### Header
- **App Icon**: The top left corner displays the Solvance logo.
- **User Information**: The user's name is "Sunil Shantha" with a unique identifier "NIC: 200012345678".
- **Status**: The status is "Active".
- **Notification**: There is a notification icon with a bell symbol, indicating there are notifications.

### Main Interface
#### Microgrid Telemetry Node
- **Solar Generation**: 14.2 kW
- **Battery Reserve**: 88% (48.4 kWh)
- **Hub Discharge Rate**: Optimal (4.8 kW outflow)
- **Hub Discharge Rate**: 567.0 kWh (180 kg CO2 saved)

####


In [ ]:
#check if gpu used
import torch

print("Model device:", next(model.parameters()).device)
print("GPU memory allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
print("GPU memory reserved:", torch.cuda.memory_reserved() / 1024**3, "GB")

In [ ]:
#check versions mismatch
import transformers
import torchvision
import torch

print("transformers:", transformers.__version__)
print("torchvision:", torchvision.__version__)
print("torch:", torch.__version__)

In [ ]:
#check if model is library CUDA is being used to use GPU
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

In [ ]:
#identify project root path
import os

print("project_root =", project_root)
print("cwd =", os.getcwd())
print("Files in project_root:")
for f in os.listdir(project_root):
    print("  ", f)

In [ ]:
import sys
print(sys.executable)

In [6]:
#test run json report output
import re
import json 
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from PIL import Image
import torch

model_path = r"D:\university\Resesarch\RD\models\Qwen2-VL-2B-Instruct"

print(f"Model device: {next(model.parameters()).device}")
print("Loading model on GPU...")
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,   # ← faster on GPU
    device_map="cuda"              # ← use your RTX 3050
)
processor = AutoProcessor.from_pretrained(model_path)

json_prompt = """Analyze this mobile app screenshot for non-crash UI defects (overlapping elements, clipping, missing elements, frozen states, broken layouts).

CRITICAL RULES:
1. If the screen looks normal with no visible defects, output overall_status="clean" and a single defect with class="none".
2. NEVER invent defects. Only report what you can clearly see.
3. Bounding boxes must reflect the ACTUAL pixel location of the defect. If unsure, set bounding_box to null.
4. Do not copy any example values.

Output STRICTLY as valid JSON. No markdown, no explanations.

Schema:
{
  "overall_status": "defective" or "clean",
  "defects": [
    {
      "defect_class": "overlapping_elements" | "clipping" | "missing_elements" | "frozen_state" | "broken_layout" | "none",
      "confidence": 0.0 to 1.0,
      "bounding_box": {"x": int, "y": int, "width": int, "height": int} or null,
      "description": "Specific visual evidence you observed"
    }
  ]
}
"""
img = Image.open(r"D:\university\Resesarch\J26-SE-334\Component3(local vision oracle)\notebooks\opening_screen.png")
message = [
    {
        "role":"user",
        "content": [
            {"type":"image","image":img},
            {"type":"text","text":json_prompt}
        ]
    }
]

print("Generating json Report...")
text = processor.apply_chat_template(message,tokenize=False,add_generation_prompt=True)
inputs = processor(text=[text],images=[img],return_tensors="pt").to("cuda")

output_ids = model.generate(**inputs,max_new_tokens=256,do_sample=False)
raw_result = processor.batch_decode(output_ids,skip_special_tokens=True)[0]

assistant_response = raw_result.split("assistant\n")[-1].strip()
print("\n Output:")
print(assistant_response)

json_match = re.search(r'\{.*\}',assistant_response,re.DOTALL)

if json_match:
    json_string = json_match.group(0)
    verdict_payload = json.loads(json_string)
    
    print("\n--- Successfully Parsed JSON ---")
    print(json.dumps(verdict_payload, indent=2))
else:
    print("\n--- Warning: Could not find valid JSON in the output ---")

Model device: cuda:0
Loading model on GPU...


Loading checkpoint shards: 100%|██████████| 2/2 [00:11<00:00,  5.88s/it]


Generating json Report...

 Output:
```json
{
  "overall_status": "defective",
  "defects": [
    {
      "defect_class": "overlapping_elements",
      "confidence": 0.0,
      "bounding_box": null,
      "description": "The 'Grid Sync' badge is overlapping with the 'Grid Sync: 99.8%' label."
    }
  ]
}
```

--- Successfully Parsed JSON ---
{
  "overall_status": "defective",
  "defects": [
    {
      "defect_class": "overlapping_elements",
      "confidence": 0.0,
      "bounding_box": null,
      "description": "The 'Grid Sync' badge is overlapping with the 'Grid Sync: 99.8%' label."
    }
  ]
}
